# F3 — Clasificación con DistilBERT

**Objetivo**: Extraer embeddings contextuales con DistilBERT y entrenar un clasificador.
**Target**: 3 clases (0=Negativo, 1=Neutro, 2=Positivo).
**MLflow**: Tracking de parámetros y métricas.
**Salida**: Reporte JSON en `reports/` + embeddings para RAG (F4).

## 1. Montar Google Drive

In [ ]:
try:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive"
PARQUET_PATH = f"{DRIVE_BASE}/office_products_balanced.parquet"
MLFLOW_PATH = f"{DRIVE_BASE}/mlruns"
EMBEDDINGS_PATH = f"{DRIVE_BASE}/distilbert_embeddings_sample.npy"
LABELS_PATH = f"{DRIVE_BASE}/distilbert_labels_sample.npy"

print(f"Parquet: {PARQUET_PATH}")
print(f"MLflow:  {MLFLOW_PATH}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
    print('Drive montado correctamente')


### Configuración de rutas

Todo el almacenamiento pesado va en Google Drive: el Parquet con 2.5M de registros, los modelos de MLflow, y los embeddings para F4. El notebook asume que el EDA ya se ejecutó y generó office_products_balanced.parquet.

## 2. Instalar dependencias

In [ ]:
!pip install -q polars torch transformers scikit-learn mlflow tqdm matplotlib seaborn
!pip install -q sentencepiece  # required for some tokenizers
import polars as pl
import numpy as np
import torch
import gc
import os
import json


## 3. Cargar datos balanceados

In [ ]:
df = pl.read_parquet(PARQUET_PATH)
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print(df.columns)
print(df['sentiment'].value_counts().sort('sentiment'))


### Datos balanceados en 3 clases

El dataset tiene 3 clases de sentimiento (0=Negativo, 1=Neutro, 2=Positivo) con 500k muestras cada una, balanceadas desde el EDA. Se usa una muestra estratificada de 100k para mantener la proporcion de clases y hacer el proceso viable en GPU T4 (12GB VRAM).

## 4. Muestreo estratificado

In [ ]:
SAMPLE_SIZE = 100_000
BATCH_SIZE = 256
MAX_LENGTH = 128
RANDOM_STATE = 42

dfs = []
for s in [0, 1, 2]:
    sub = df.filter(pl.col('sentiment') == s)
    n = min(SAMPLE_SIZE // 3, sub.shape[0])
    dfs.append(sub.sample(n=n, seed=RANDOM_STATE))

df_sample = pl.concat(dfs).sample(frac=1.0, seed=RANDOM_STATE)
print(f"Sample: {df_sample.shape}")
print(df_sample['sentiment'].value_counts().sort('sentiment'))


## 5. Train/Val/Test split

In [ ]:
from sklearn.model_selection import train_test_split

texts = df_sample['text'].to_list()
labels = df_sample['sentiment'].to_numpy()

X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.15, random_state=RANDOM_STATE, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85  # = ~0.176: 15% del total original para validation, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


### Partición train/val/test

70% entrenamiento, 15% validación, 15% prueba. La proporción 0.15/0.85 = ~0.176 redimensiona el validation set para que sea el 15% del total original. El split es estratificado: cada subconjunto mantiene la misma proporción de clases que el dataset completo.

## 6. Tokenización y Embeddings con DistilBERT

In [ ]:
try:
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"Device: {device}")


### DistilBERT: modelo pre-entrenado congelado

Se usa DistilBERT (distilbert-base-uncased) en modo inference (model.eval(), torch.no_grad()). No se fine-tune el transformer por dos razones: (1) extraer embeddings de un modelo congelado es más rápido y consume menos VRAM; (2) los embeddings contextuales de DistilBERT ya capturan información semántica suficiente para un clasificador lineal. El embedding del token [CLS] (posición 0 del último hidden state) se usa como representación de toda la reseña.

In [ ]:
def extract_embeddings(texts, model, tokenizer, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            outputs = model(**encoded)
        # [CLS] token embedding
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        del encoded, outputs
        if i % (batch_size * 10) == 0:
            gc.collect()
    return np.vstack(all_embeddings)


In [ ]:
# Extraer embeddings (esto toma varios minutos en T4)
print("Extrayendo embeddings TRAIN...")
X_train_emb = extract_embeddings(X_train, model, tokenizer)
print(f"Train embeddings: {X_train_emb.shape}")

print("Extrayendo embeddings VAL...")
X_val_emb = extract_embeddings(X_val, model, tokenizer)
print(f"Val embeddings: {X_val_emb.shape}")

print("Extrayendo embeddings TEST...")
X_test_emb = extract_embeddings(X_test, model, tokenizer)
print(f"Test embeddings: {X_test_emb.shape}")


## 7. Entrenar clasificador sobre embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
import time

clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)

start = time.time()
clf.fit(X_train_emb, y_train)
training_time = time.time() - start
print(f"Entrenado en {training_time:.1f}s")


### Clasificador sobre embeddings

Se entrena una regresión logística (LogisticRegression) sobre los embeddings de 768 dimensiones. La regresión logística es computacionalmente eficiente y sirve como baseline sólido para evaluar si los embeddings de DistilBERT capturan información de sentimiento. Si el F1-macro supera ~0.60, los embeddings son informativos.

In [ ]:
# Evaluación
y_pred = clf.predict(X_test_emb)

metrics = {
    'model_name': 'DistilBERT + LogisticRegression',
    'model_type': 'distilbert_base_uncased + logreg',
    'sample_size': SAMPLE_SIZE,
    'batch_size': BATCH_SIZE,
    'max_length': MAX_LENGTH,
    'embedding_dim': X_train_emb.shape[1],
    'training_time_seconds': round(training_time, 2),
    'f1_macro': round(f1_score(y_test, y_pred, average='macro'), 4),
    'precision_macro': round(precision_score(y_test, y_pred, average='macro'), 4),
    'recall_macro': round(recall_score(y_test, y_pred, average='macro'), 4),
    'accuracy': round(accuracy_score(y_test, y_pred), 4),
    'class_labels': ['Negativo', 'Neutro', 'Positivo'],
    'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
}

for k, v in metrics.items():
    if k != 'confusion_matrix':
        print(f"  {k}: {v}")


### Interpretación de resultados

Las métricas reportadas (F1-macro, precisión, recall y accuracy) permiten evaluar el desempeño del clasificador. El F1-macro promedia las tres clases por igual, por lo que no favorece a clases mayoritarias. La matriz de confusión muestra dónde se confunde el modelo: si hay sesgo hacia una clase o si ciertos pares (ej: Neutro ↔ Positivo) son más difíciles de separar.

## 8. MLflow Tracking

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"file://{MLFLOW_PATH}")
mlflow.set_experiment("distilbert_office_products")

with mlflow.start_run():
    mlflow.log_params({
        "model_name": MODEL_NAME,
        "sample_size": SAMPLE_SIZE,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
        "classifier": "LogisticRegression",
        "embedding_dim": X_train_emb.shape[1],
    })
    mlflow.log_metrics({
        "f1_macro": metrics['f1_macro'],
        "precision_macro": metrics['precision_macro'],
        "recall_macro": metrics['recall_macro'],
        "accuracy": metrics['accuracy'],
        "training_time_seconds": metrics['training_time_seconds'],
    })
    mlflow.sklearn.log_model(clf, "classifier")
    print(f"MLflow run ID: {mlflow.active_run().info.run_id}")

print(f"MLflow runs en: {MLFLOW_PATH}")


### MLflow tracking

Cada ejecución queda registrada en MLflow con parámetros (model_name, sample_size, batch_size, etc.) y métricas (F1, precisión, recall, accuracy, tiempo de entrenamiento). El tracking URI apunta a Google Drive para persistencia entre sesiones de Colab. El modelo entrenado se guarda como artefacto para reutilización.

## 9. Exportar métricas a JSON

In [ ]:
os.makedirs('/content/reports', exist_ok=True)
with open('/content/reports/metrics_distilbert.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Exportado: /content/reports/metrics_distilbert.json")

# Copiar al repo si está montado
if os.path.exists('/content/proyecto-integrador-ml-amazon-reviews'):
    import shutil
    shutil.copy(
        '/content/reports/metrics_distilbert.json',
        '/content/proyecto-integrador-ml-amazon-reviews/reports/metrics_distilbert.json'
    )
    print("Copiado a reports/ en el repo")


## 10. Guardar embeddings para F4 (RAG)

### Embeddings para F4 (RAG)

Se guardan 10k embeddings (muestra estratificada) junto con sus etiquetas. Estos embeddings servirán en F4 para construir un sistema de Retrieval-Augmented Generation: dada una consulta, buscar reseñas similares por similitud coseno y usar ese contexto para responder. Los embeddings se guardan en Drive porque el archivo .npy pesa ~30MB.

## 11. Visualizaciones

In [ ]:
# Matriz de confusión
import matplotlib.pyplot as plt
import seaborn as sns

cm = np.array(metrics['confusion_matrix'])
labels_names = metrics['class_labels']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_names, yticklabels=labels_names)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de Confusión - DistilBERT')
plt.tight_layout()
plt.show()


### Análisis de la matriz de confusión

La diagonal principal muestra los aciertos por clase. Los errores fuera de la diagonal indican confusiones típicas: si hay más errores entre clases adyacentes (Neg-Neu, Neu-Pos) es esperable porque la frontera entre sentimientos vecinos es difusa. Errores entre Neg-Pos serían más graves y sugerirían problemas en los embeddings.

In [ ]:
# TSNE de embeddings
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
emb_tsne = tsne.fit_transform(emb_viz[:5000])  # subset para velocidad

plt.figure(figsize=(10, 8))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
for label in [0, 1, 2]:
    mask = labels_viz[:5000] == label
    plt.scatter(emb_tsne[mask, 0], emb_tsne[mask, 1],
                c=colors[label], label=labels_names[label], alpha=0.6, s=10)
plt.title('t-SNE de Embeddings DistilBERT (5k muestras)')
plt.legend()
plt.tight_layout()
plt.show()


### t-SNE: visualización de embeddings

El t-SNE proyecta los embeddings 768D a 2D para inspección visual. Si las clases se separan en clusters distinguibles, los embeddings capturan información de sentimiento. Si se superponen completamente, el modelo no logra discriminar. El solapamiento entre clases vecinas (Neg-Neu, Neu-Pos) es esperable.

In [ ]:
# Liberar memoria
del df, df_sample, model, tokenizer
del X_train_emb, X_val_emb, X_test_emb, y_train, y_val, y_test
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
